# 日本株 ネットキャッシュ比率スクリーナー

**スクリーニング条件:**
- ネットキャッシュ = 流動資産 + 投資有価証券×70% − 負債合計
- ネットキャッシュ比率 = ネットキャッシュ ÷ 時価総額 > **1.0**

**データソース:** JPX公式銘柄リスト + Yahoo Finance (yfinance)

In [ ]:
# Cell 1: ライブラリインストール・インポート
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['yfinance', 'pandas', 'requests', 'tqdm', 'openpyxl']:
    install(pkg)

import yfinance as yf
import pandas as pd
import requests
import pickle
import time
import os
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import display

print('ライブラリ読み込み完了')

In [ ]:
# Cell 2: JPX銘柄リスト取得
JPX_URL = 'https://www.jpx.co.jp/markets/statistics-equities/misc/tvdivq0000001vg2-att/data_j.xls'
LOCAL_XLS = 'data_j.xls'

if not Path(LOCAL_XLS).exists():
    print('JPX銘柄リストをダウンロード中...')
    r = requests.get(JPX_URL, headers={'User-Agent': 'Mozilla/5.0'})
    r.raise_for_status()
    with open(LOCAL_XLS, 'wb') as f:
        f.write(r.content)
    print(f'ダウンロード完了: {LOCAL_XLS}')
else:
    print(f'キャッシュ済み: {LOCAL_XLS}')

jpx_df = pd.read_excel(LOCAL_XLS, dtype={'コード': str})
print(f'列名: {list(jpx_df.columns)}')
print(f'総銘柄数: {len(jpx_df)}')
display(jpx_df.head())

In [ ]:
# Cell 2b: ティッカーリスト生成（市場区分フィルタ可能）

# 対象市場を選択（コメントアウトで全市場 or 特定市場に絞れる）
TARGET_MARKETS = None  # None = 全市場
# TARGET_MARKETS = ['プライム（内国株式）']  # プライムのみ
# TARGET_MARKETS = ['プライム（内国株式）', 'スタンダード（内国株式）']  # プライム+スタンダード

# 市場区分の列名を確認して設定
market_col = '市場・商品区分'  # JPXファイルの列名

if TARGET_MARKETS:
    filtered = jpx_df[jpx_df[market_col].isin(TARGET_MARKETS)].copy()
else:
    # ETF・REIT等を除いた内国普通株式のみ
    filtered = jpx_df[jpx_df[market_col].str.contains('内国株式', na=False)].copy()

# ティッカーリスト生成（4桁コード → XXXX.T）
filtered['ticker'] = filtered['コード'].str.zfill(4) + '.T'

print(f'対象銘柄数: {len(filtered)}')
print(f'市場区分内訳:')
print(filtered[market_col].value_counts())

tickers = filtered['ticker'].tolist()
ticker_info = filtered.set_index('ticker')[['コード', '銘柄名', market_col, '33業種区分']].copy()

In [ ]:
# Cell 3: 財務データ取得（キャッシュ付き・レートリミット対策済み）
CACHE_FILE = 'net_cash_cache.pkl'
SLEEP_SEC = 0.5   # リクエスト間の待機秒数（レートリミット対策）
BATCH_SIZE = 100  # 何銘柄ごとにキャッシュ保存するか

def get_value(df, candidates):
    """バランスシートDFから複数候補列名のうち最初に見つかった値を返す"""
    for name in candidates:
        if name in df.index:
            val = df.loc[name].iloc[0]  # 最新年度
            if pd.notna(val):
                return float(val)
    return None

CURRENT_ASSETS_KEYS = [
    'Current Assets', 'Total Current Assets', 'CurrentAssets'
]
INVEST_SEC_KEYS = [
    'Available For Sale Securities', 'Long Term Investments',
    'Investmentin Financial Assets', 'Other Investments',
    'Investment Securities', 'Investments And Advances'
]
TOTAL_LIAB_KEYS = [
    'Total Liabilities Net Minority Interest', 'Total Liab',
    'Total Liabilities', 'TotalLiabilitiesNetMinorityInterest'
]

def fetch_stock_data(ticker):
    """1銘柄の財務データを取得して辞書で返す"""
    try:
        stock = yf.Ticker(ticker)
        bs = stock.balance_sheet
        if bs is None or bs.empty:
            return None
        
        info = stock.info
        market_cap = info.get('marketCap') or info.get('market_cap')
        if not market_cap:
            return None
        
        current_assets = get_value(bs, CURRENT_ASSETS_KEYS)
        invest_sec = get_value(bs, INVEST_SEC_KEYS) or 0.0
        total_liab = get_value(bs, TOTAL_LIAB_KEYS)
        
        if current_assets is None or total_liab is None:
            return None
        
        return {
            'current_assets': current_assets,
            'investment_securities': invest_sec,
            'total_liabilities': total_liab,
            'market_cap': float(market_cap),
        }
    except Exception:
        return None

# キャッシュ読み込み
if Path(CACHE_FILE).exists():
    with open(CACHE_FILE, 'rb') as f:
        cache = pickle.load(f)
    print(f'キャッシュ読み込み済み: {len(cache)} 銘柄')
else:
    cache = {}
    print('キャッシュなし。新規取得を開始します。')

remaining = [t for t in tickers if t not in cache]
failed = []
print(f'取得済み: {len(cache)} / 残り: {len(remaining)} 銘柄')

for i, ticker in enumerate(tqdm(remaining, desc='財務データ取得')):
    result = fetch_stock_data(ticker)
    if result:
        cache[ticker] = result
    else:
        failed.append(ticker)
    
    time.sleep(SLEEP_SEC)
    
    # バッチごとに中間保存
    if (i + 1) % BATCH_SIZE == 0:
        with open(CACHE_FILE, 'wb') as f:
            pickle.dump(cache, f)

# 最終保存
with open(CACHE_FILE, 'wb') as f:
    pickle.dump(cache, f)

print(f'\n取得成功: {len(cache)} 銘柄 / 取得失敗: {len(failed)} 銘柄')
if failed:
    print(f'失敗銘柄（先頭20件）: {failed[:20]}')

In [ ]:
# Cell 4: ネットキャッシュ計算・フィルタリング
rows = []
for ticker, data in cache.items():
    row = {'ticker': ticker}
    row.update(data)
    rows.append(row)

df = pd.DataFrame(rows).set_index('ticker')

# ネットキャッシュ計算
df['net_cash'] = (
    df['current_assets']
    + df['investment_securities'] * 0.7
    - df['total_liabilities']
)
df['net_cash_ratio'] = df['net_cash'] / df['market_cap']

# 銘柄情報と結合
df = df.join(ticker_info, how='left')

# フィルタリング: ネットキャッシュ比率 > 1
NET_CASH_RATIO_THRESHOLD = 1.0
result = df[df['net_cash_ratio'] > NET_CASH_RATIO_THRESHOLD].copy()
result = result.sort_values('net_cash_ratio', ascending=False)

print(f'全取得銘柄: {len(df)} 社')
print(f'ネットキャッシュ比率 > {NET_CASH_RATIO_THRESHOLD} の銘柄: {len(result)} 社')

In [ ]:
# Cell 5: 結果表示・CSVエクスポート
def fmt_oku(val):
    """円を億円表示に変換"""
    return f'{val/1e8:,.1f}億円' if pd.notna(val) else '-'

display_cols = {
    'コード': 'コード',
    '銘柄名': '銘柄名',
    '市場・商品区分': '市場',
    '33業種区分': '業種',
    'current_assets': '流動資産',
    'investment_securities': '投資有価証券',
    'total_liabilities': '負債合計',
    'net_cash': 'ネットキャッシュ',
    'market_cap': '時価総額',
    'net_cash_ratio': 'NC比率',
}

show = result.reset_index()[list(display_cols.keys())].rename(columns=display_cols).copy()

for col in ['流動資産', '投資有価証券', '負債合計', 'ネットキャッシュ', '時価総額']:
    show[col] = show[col].apply(fmt_oku)

show['NC比率'] = show['NC比率'].apply(lambda x: f'{x:.2f}')

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
display(show)

# CSVエクスポート
csv_path = 'net_cash_result.csv'
result.reset_index().to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'\nCSV保存完了: {csv_path}')